模糊回归模型的数据是所有年份的（X，y）<br>
熵权+综合评判模型的数据是2023年（X_2023,y）

<h1><font color="blue">模糊回归</font></h1>
mohu_model.py

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import sys
import os
from sklearn.linear_model import LinearRegression

# 获取 project 目录（上一级）
project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_path)
from models.mohu_model import IterativeMohuDecision
from models.model_validation import validate_fuzzy_model

In [ ]:
#重新加载mohu_model.py
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
df = pd.read_excel('../data/city_data.xlsx')

In [ ]:
y = df["inno"].values
X = df.drop(columns=["year", "city", "inno"]).values
#标准化
X_mean,X_std = X.mean(axis=0),X.std(axis=0)
y_mean, y_std = y.mean(), y.std()

X = (X - X_mean) / X_std
y = (y - y_mean) / y_std
#最终使用的数据是标准化后的X和y

<h1>一、建模</h1>

In [ ]:
n = X.shape[1]
number = X.shape[0]

model = IterativeMohuDecision(
    n=n,
    number=number,
    tol=1e-6,
    max_iter=50
)

model.setdatax(X)
model.setdatay(y)
model.setdataxishu(np.zeros(n + 1))

In [ ]:
#拟合
beta = model.fit()
print("最终模糊回归系数：")
print(beta)
# ✅ 判断标准：没有 NaN；没有爆成极大值；多次运行结果稳定

最终模糊回归系数：
[-0.03085755  0.27793805  0.1890206   0.10142322 -0.00457045 -0.26476851
  0.05483536  0.27277228  0.10740055  0.19901718  0.15961061 -0.09155732
  0.06420456 -0.18887027  0.06447467  0.10214021  0.15192407  0.01942734
  0.07063777]


<h2>1.拟合能力（正常数据）</h2>

In [ ]:
#定量指标（必须有）
y_pred = model.predict(X)
#模糊回归
rmse = np.sqrt(np.mean((y - y_pred) ** 2))
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - ss_res / ss_tot
print("RMSE =", rmse)
print("R2   =", r2)
#✅ 你期望看到的情况：RMSE < 1（标准化后）；R2 > 0.6 即可接受，>0.7 很好

RMSE = 0.33309152876426296
R2   = 0.8890500334654862


In [ ]:
#与普通线性回归对比（普通最小二乘回归）OLS
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X, y)

y_lr = lr.predict(X)

rmse_lr = np.sqrt(np.mean((y - y_lr) ** 2))
r2_lr = 1 - np.sum((y - y_lr) ** 2) / np.sum((y - y.mean()) ** 2)

print("OLS RMSE =", rmse_lr)
print("OLS R2   =", r2_lr)
#如果模糊回归 RMSE ≤ OLS或 R² 更稳定，说明模型不比传统方法差

OLS RMSE = 0.32335109115973865
OLS R2   = 0.8954440718458063


解读：两个的RMSE和R^2都差不多，说明模糊回归模型拟合能力良好,没有明显的性能损失。

<h2>2.系数稳定性</h2>

In [ ]:
def add_outliers(X, y, ratio=0.15, scale=5):
    X_new = X.copy()
    y_new = y.copy()
    n_out = int(len(y) * ratio)
    idx = np.random.choice(len(y), n_out, replace=False)

    X_new[idx] *= scale
    y_new[idx] *= scale
    return X_new, y_new

In [ ]:
X_bad, y_bad = add_outliers(X, y, ratio=0.2, scale=5)

# 模糊回归
model_bad = IterativeMohuDecision(n, number)
model_bad.setdatax(X_bad)
model_bad.setdatay(y_bad)
model_bad.setdataxishu(np.zeros(n + 1))
beta_bad = model_bad.fit()

delta_mohu = np.linalg.norm(beta - beta_bad)

# OLS
lr_bad = LinearRegression()
lr_bad.fit(X_bad, y_bad)
delta_ols = np.linalg.norm(lr.coef_ - lr_bad.coef_)

print("模糊回归变化 =", delta_mohu)
print("OLS 变化 =", delta_ols)


模糊回归变化 = 0.29623919952171746
OLS 变化 = 0.6186613626432698


在“轻微扰动”下，两者都稳定。OLS 略占优（这很正常），用来证明没有作弊。

<h2>3.强异常值鲁棒性验证</h2>
理想结果：delta (模糊回归) << delta_lr (OLS)

In [ ]:
#人为制造异常样本
def add_outliers(X, y, ratio=0.15, scale=5):
    X_new = X.copy()
    y_new = y.copy()
    n_out = int(len(y) * ratio)
    idx = np.random.choice(len(y), n_out, replace=False)

    X_new[idx] *= scale
    y_new[idx] *= scale
    return X_new, y_new

In [ ]:
X_bad, y_bad = add_outliers(X, y, ratio=0.2, scale=5)

# 模糊回归
model_bad = IterativeMohuDecision(n, number)
model_bad.setdatax(X_bad)
model_bad.setdatay(y_bad)
model_bad.setdataxishu(np.zeros(n + 1))
beta_bad = model_bad.fit()

delta_mohu = np.linalg.norm(beta - beta_bad)

# OLS
lr_bad = LinearRegression()
lr_bad.fit(X_bad, y_bad)
delta_ols = np.linalg.norm(lr.coef_ - lr_bad.coef_)

print("模糊回归变化 =", delta_mohu)
print("OLS 变化 =", delta_ols)
#理想结果：
# 模糊回归变化 = 0.1 ~ 0.3
# OLS 变化     = 1.0 ~ 5.0

模糊回归变化 = 0.37240869705928786
OLS 变化 = 0.6568756475160444


当样本中引入明显异常值时，OLS 模型参数变化更为显著。模糊回归对异常样本的敏感性更低

In [ ]:
#异常比例敏感性
ratios = [0.05, 0.1, 0.2, 0.3]
for r in ratios:
    X_bad, y_bad = add_outliers(X, y, ratio=r, scale=5)

    model_bad = IterativeMohuDecision(n, number)
    model_bad.setdatax(X_bad)
    model_bad.setdatay(y_bad)
    model_bad.setdataxishu(np.zeros(n + 1))
    beta_bad = model_bad.fit()
    delta_m = np.linalg.norm(beta - beta_bad)

    lr_bad = LinearRegression()
    lr_bad.fit(X_bad, y_bad)
    delta_l = np.linalg.norm(lr.coef_ - lr_bad.coef_)

    print(f"异常比例 {r:.0%}: 模糊={delta_m:.3f}, OLS={delta_l:.3f}")
# 异常比例 ↑。。。OLS 系数变化 ↑ 更快。。。。模糊回归 ↑ 较慢


异常比例 5%: 模糊=0.225, OLS=0.370
异常比例 10%: 模糊=0.202, OLS=0.315
异常比例 20%: 模糊=0.558, OLS=0.586
异常比例 30%: 模糊=0.417, OLS=0.860


<h1>验证整合</h1>model_validation.py

In [111]:
import pandas as pd
import numpy as np
import sys
import os
from sklearn.linear_model import LinearRegression

# 获取 project 目录（上一级）
project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_path)
from models.mohu_model import IterativeMohuDecision
from models.model_validation import validate_fuzzy_model


In [112]:
# df = pd.read_excel('../data/city_data.xlsx')

# y = df["inno"].values
# X = df.drop(columns=["year", "city", "inno"]).values

In [113]:
n = X.shape[1]
number = X.shape[0]
fuzzy_model = IterativeMohuDecision(
    n=n,
    number=number,
    tol=1e-6,
    max_iter=50
)
ols_model = LinearRegression()

results = validate_fuzzy_model(X, y, fuzzy_model, ols_model)

for k, v in results.items():
    print(f"{k}: {v:.4f}")



RMSE_fuzzy: 0.3331
RMSE_OLS: 0.3234
R2_fuzzy: 0.8891
R2_OLS: 0.8954
delta_fuzzy: 0.1884
delta_OLS: 2.6579


<h1><font color="blue">熵权法</font></h1>

<h2>1.指标正向化</h2>
在平台里可以用勾选框、字段配置来决定每一列用哪种

In [114]:
import numpy as np

def normalize(X, indicator_type):
    """
    X: (m, n) 原始数据
    indicator_type: list, 每列是 'positive' 或 'negative'
    """
    X = np.asarray(X, dtype=float)
    X_norm = np.zeros_like(X)

    for j, t in enumerate(indicator_type):
        min_, max_ = X[:, j].min(), X[:, j].max()
        if abs(max_ - min_) < 1e-12:
            X_norm[:, j] = 0
        elif t == 'positive':
            X_norm[:, j] = (X[:, j] - min_) / (max_ - min_)
        else:
            X_norm[:, j] = (max_ - X[:, j]) / (max_ - min_)
    return X_norm

In [115]:
def auto_indicator_type(n, default='positive'):
    return [default] * n

<h2>2.计算熵权</h2>

In [116]:
def entropy_weight(X):
    """
    X: 已经正向化后的矩阵 (m, n)
    return: 权重向量 (n,)
    """
    m, n = X.shape
    # 避免 log(0)
    P = X / (X.sum(axis=0) + 1e-12)
    # 熵值
    E = -np.sum(P * np.log(P + 1e-12), axis=0) / np.log(m)
    # 信息效用值
    d = 1 - E
    # 权重
    w = d / d.sum()
    return w

<h2>3.构造模糊综合评判的“等级体系”</h2>

In [117]:
V = ['很低', '较低', '中等', '较高', '很高']

<h2>4.模糊隶属度函数</h2>

In [118]:
#三角隶属度函数
def triangular_membership(x, a, b, c):
    if x <= a or x >= c:
        return 0.0
    elif a < x <= b:
        return (x - a) / (b - a)
    else:
        return (c - x) / (c - b)


<h2>5.构造模糊关系矩阵R</h2>

In [119]:
# 对应 V = ['很低', '较低', '中等', '较高', '很高']
levels = [
    (0.0, 0.0, 0.25),
    (0.0, 0.25, 0.5),
    (0.25, 0.5, 0.75),
    (0.5, 0.75, 1.0),
    (0.75, 1.0, 1.0)
]

In [120]:
#单个样本的R矩阵
def build_fuzzy_relation(x_norm_row, levels):
    """
    x_norm_row: 单个样本 (n,)
    return: R (n, k)
    """
    n = len(x_norm_row)
    k = len(levels)
    R = np.zeros((n, k))

    for i in range(n):
        for j, (a, b, c) in enumerate(levels):
            R[i, j] = triangular_membership(x_norm_row[i], a, b, c)

        # 归一化，防止数值误差
        s = R[i].sum()
        if s > 0:
            R[i] /= s
    return R

<h2>6.模糊综合评价</h2>

In [121]:
#加权平均型算子
def fuzzy_comprehensive_evaluation(w, R):
    """
    w: 指标权重 (n,)
    R: 模糊关系矩阵 (n, k)
    """
    B = w @ R
    return B / B.sum()


<h2>7.可解释的最终结果</h2>

In [122]:
#等级判定
def grade_decision(B, V):
    idx = np.argmax(B)
    return V[idx]

#数值化得分
def score_from_B(B):
    # 例如 1~5 分
    scores = np.arange(1, len(B) + 1)
    return np.dot(B, scores)


In [123]:
df_2023 = df[df['year'] == 2023]
# X_2023 = df_2023.drop(columns=["year", "city", "inno"]).values
#使用的不是标准化的X，y。没有与第一个模型的xy值对应

In [124]:
#先提取2023年的原始数据，再标准化
X_2023_raw = df_2023.drop(columns=["year", "city", "inno"]).values
# 使用之前计算的标准化的均值X_mean和标准差进行标准化，得到X_2023：X_2023 = (X_2023_raw - X_mean) / X_std。
# 要么使用标准化的X_2023: 
# 要么使用正向归一化的X_2023_norm。（我的选择）
# 同时使用标准化+正向归一化，可能会扭曲数据。

In [125]:
indicator_type = auto_indicator_type(X_2023_raw.shape[1])
X_2023_norm = normalize(X_2023_raw, indicator_type)#或混合


In [126]:
print(indicator_type)

['positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive', 'positive']


In [127]:
weights = entropy_weight(X_2023_norm) # 熵权法->指标权重
print(weights)
print("sum的值为",weights.sum())  # sum必须等于 1

[0.05561794 0.02991734 0.03673409 0.03094216 0.066016   0.0518807
 0.06213474 0.02600849 0.05257993 0.06974208 0.04708384 0.02827188
 0.03807626 0.11981046 0.08179668 0.09003308 0.09997722 0.01337708]
sum的值为 1.0


In [128]:
score = X_2023_norm @ weights

In [129]:
results = []
for i in range(X_2023_norm.shape[0]):
    R = build_fuzzy_relation(X_2023_norm[i], levels)
    B = fuzzy_comprehensive_evaluation(weights, R)

    results.append({
        'B': B,
        'grade': grade_decision(B, V),
        'score': score_from_B(B)
    })

In [130]:
print(results)

[{'B': array([0.15088817, 0.31665098, 0.20897285, 0.1662101 , 0.15727791]), 'grade': '较低', 'score': 2.8623385919663304}, {'B': array([0.22218485, 0.44781123, 0.14796733, 0.09168784, 0.09034875]), 'grade': '较低', 'score': 2.380204395090678}, {'B': array([0.44161691, 0.34367333, 0.14864821, 0.04300447, 0.02305708]), 'grade': '很低', 'score': 1.8622114634706244}, {'B': array([0.34942831, 0.43325276, 0.17137817, 0.02707017, 0.0188706 ]), 'grade': '较低', 'score': 1.9327019900257723}, {'B': array([0.13353403, 0.46737871, 0.28756019, 0.07408277, 0.0374443 ]), 'grade': '较低', 'score': 2.414524610136426}, {'B': array([0.40342897, 0.40497562, 0.1415366 , 0.03786165, 0.01219716]), 'grade': '较低', 'score': 1.8504224208508486}, {'B': array([0.5020477 , 0.26308093, 0.17671259, 0.03804483, 0.02011396]), 'grade': '很低', 'score': 1.8110964276742885}, {'B': array([0.35049558, 0.44399042, 0.09466901, 0.0695075 , 0.04133749]), 'grade': '较低', 'score': 2.007200902423987}, {'B': array([0.4637384 , 0.29055335, 0.121

In [131]:
df_results = pd.DataFrame(results)
print(df_results)

                                                    B grade     score
0   [0.15088817131462876, 0.3166509767336404, 0.20...    较低  2.862339
1   [0.2221848514265482, 0.44781123080445256, 0.14...    较低  2.380204
2   [0.44161691460922703, 0.34367333203656697, 0.1...    很低  1.862211
3   [0.3494283067933552, 0.43325276112877786, 0.17...    较低  1.932702
4   [0.13353402953835491, 0.4673787080349168, 0.28...    较低  2.414525
5   [0.4034289713385614, 0.40497561597834786, 0.14...    较低  1.850422
6   [0.5020476952804461, 0.2630809267474505, 0.176...    很低  1.811096
7   [0.35049557892562827, 0.4439904202378354, 0.09...    较低  2.007201
8   [0.4637384004530059, 0.29055334547252265, 0.12...    很低  1.996224
9   [0.2275026217278005, 0.3563023128991698, 0.279...    较低  2.353938
10  [0.12122456738623089, 0.18745441802671928, 0.3...    中等  3.025517
11  [0.30731954014380786, 0.2625243069337983, 0.14...    很低  2.484972
12  [0.28632293880802284, 0.4326585696709353, 0.14...    较低  2.144492


In [132]:
df_results['year'] = df_2023['year'].values
df_results['city'] = df_2023['city'].values
df_results['inno'] = df_2023['inno'].values

In [133]:
df_results.to_excel('../data/city_fuzzy_evaluation_results.xlsx', index=False)

查看模型评价后的排名与实际排名

In [134]:
df_2023=df[df['year']==2023]
df_2023 = df_2023.sort_values(by='inno', ascending=False)
print(df_2023['city'])

120    深圳
10     北京
21     上海
43     苏州
109    广州
54     杭州
87     武汉
142    成都
32     南京
65     宁波
76     合肥
131    重庆
98     长沙
Name: city, dtype: object


In [135]:
df_results_1 = pd.read_excel('../data/city_fuzzy_evaluation_results.xlsx')
df_results_1 = df_results_1[df_results_1['year']==2023]
df_results_1 = df_results_1.sort_values(by='score', ascending=False)
print(df_results_1['city'])

10    深圳
0     北京
11    重庆
4     杭州
1     上海
9     广州
12    成都
7     武汉
8     长沙
3     苏州
2     南京
5     宁波
6     合肥
Name: city, dtype: object


<h3>(1) 排序一致性验证</h3>
如果模型是“对的”，那么得分高的城市，不应频繁被小扰动打到很后面。排名应当整体稳定，而非剧烈震荡。

In [136]:
from scipy.stats import spearmanr

def ranking_stability(X, indicator_type, noise_ratio=0.02, trials=50):
    X_norm = normalize(X, indicator_type)
    w0 = entropy_weight(X_norm)
    base_scores = []
    for i in range(X_norm.shape[0]):
        R = build_fuzzy_relation(X_norm[i], levels)
        B = fuzzy_comprehensive_evaluation(w0, R)
        base_scores.append(score_from_B(B))
    base_rank = np.argsort(base_scores)
    correlations = []
    for _ in range(trials):
        noise = np.random.normal(0, noise_ratio, X.shape)
        X_noise = X * (1 + noise)
        Xn = normalize(X_noise, indicator_type)
        w = entropy_weight(Xn)
        scores = []
        for i in range(Xn.shape[0]):
            R = build_fuzzy_relation(Xn[i], levels)
            B = fuzzy_comprehensive_evaluation(w, R)
            scores.append(score_from_B(B))
        rank = np.argsort(scores)
        corr, _ = spearmanr(base_rank, rank)
        correlations.append(corr)
    return np.mean(correlations)


In [137]:
rank = ranking_stability(X_2023_norm, indicator_type, noise_ratio=0.02, trials=50)
print("排名稳定性指标 =", rank)

排名稳定性指标 = 0.7958241758241758


| Spearman ρ | 解释    |
| ---------- | ----- |
| > 0.9      | 非常稳定  |
| 0.8–0.9    | 稳定    |
| 0.7–0.8    | 可接受   |
| < 0.7      | 模型需谨慎 |


<h3>(2) 验证指标权重（熵权是否有意义）</h3>

In [138]:
import numpy as np

def weight_summary(w):
    return {
        "min": np.min(w),
        "max": np.max(w),
        "std": np.std(w),
        "entropy": -np.sum(w * np.log(w + 1e-12))
    }

print(weight_summary(weights))
# 最小权重 ≈ 0.01，最大权重 ≈ 0.12，标准差 ≈ 0.028。
# 没有指标“独裁”或者无用，熵权法是有效的。
# 各指标的信息熵存在一定差异，但未出现单一指标权重过大的情况，
# 指标体系结构较为均衡，熵权赋权结果具有合理性

{'min': 0.013377075076562082, 'max': 0.11981046331174797, 'std': 0.027635256816847514, 'entropy': 2.7685019262660426}


<h3>（3）综合评判结果的区分度</h3>

In [139]:
scores = np.array([r['score'] for r in results])

print("均值:", scores.mean())
print("标准差:", scores.std())
print("极差:", scores.max() - scores.min())
#在 1–5 级评分体系下：标准差 0.37 → 区分度明显；极差 1.21 → 样本间差异充分
# 说明模型确实在“拉开城市差距”

均值: 2.240449556463463
标准差: 0.3744820115512853
极差: 1.21442082182537


<h3>（4）排序稳定性（数据轻微扰动，排名是否“乱飞”）</h3>

In [140]:
def perturb_test(X, indicator_type, w, noise_level=0.05):
    X_perturbed = X * (1 + noise_level * np.random.randn(*X.shape))
    X_norm = normalize(X_perturbed, indicator_type)
    scores = []
    for i in range(X_norm.shape[0]):
        R = build_fuzzy_relation(X_norm[i], levels)
        B = fuzzy_comprehensive_evaluation(w, R)
        scores.append(score_from_B(B))

    return np.array(scores)
scores_base = scores
scores_noise = perturb_test(X_2023_norm, indicator_type, weights, noise_level=0.05)

rank_base = np.argsort(scores_base)
rank_noise = np.argsort(scores_noise)

rank_diff = np.mean(np.abs(rank_base - rank_noise))
print("平均排序变化:", rank_diff)
#1.38表明：数据加入5%左右扰动后，排名平均变化约1.38名，说明模型排名对数据扰动具有一定的鲁棒性。

平均排序变化: 1.2307692307692308


<h3>（5）与真实创新指数的一致性验证</h3>

In [141]:
from scipy.stats import spearmanr

rank_model = np.argsort(scores)
rank_inno = np.argsort(df_2023['inno'].values)  # 真实创新指数

rho, p = spearmanr(rank_model, rank_inno)
print("Spearman 相关系数:", rho)

Spearman 相关系数: -0.14285714285714285


<h1>模型二整合</h1>entropy_fce.py

In [142]:
from models.entropy_fce import EntropyFCEModel
indicator_type = ['positive'] * X_2023_norm.shape[1]#自动生成全正向指标类型(positive or negative)，默认为正向
model = EntropyFCEModel(indicator_type)
weights = model.fit(X_2023_norm)#熵权
results = model.evaluate(X_2023_norm)#模糊综合评判

for r in results[:]:
    print(r["grade"], r["score"], r["B"])


AttributeError: 'EntropyFCEModel' object has no attribute 'semantic_conf'

<h1><font color="blue">模型一+二整合</font></h1>
decision_engine.py

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import sys
import os
from sklearn.linear_model import LinearRegression

# 获取 project 目录（上一级）
project_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_path)
from models.mohu_model import IterativeMohuDecision
from models.model_validation import validate_fuzzy_model

In [ ]:
from models.decision_engine import DecisionEngine

indicator_type = ['positive'] * X.shape[1]

engine = DecisionEngine(
    regression_config={
        "n": X.shape[1],
        "number": X.shape[0]
    },
    indicator_type=indicator_type
)


result = engine.full_decision(X_eval=X_2023_norm)

#print("模糊回归系数:", result["regression_coef"])
print("熵权:", result["weights"])
print("第一个城市评价:", result["evaluation"][0])


In [ ]:
# 模糊回归用标准化数据（所有年份）
X_reg = X
y_reg = y

# 熵权+模糊综合评价用正向归一化数据（2023年）
X_eval = X_2023_norm  # 正向归一化的2023年数据

result = engine.full_decision(
    X_eval=X_eval,
    X_reg=X_reg,
    y_reg=y_reg
)

print("模糊回归系数:", result.get("regression_coef", "未执行模糊回归"))
print("熵权:", result.get("weights"))
print("第一个城市评价:", result.get("evaluation")[0])

In [ ]:
engine = DecisionEngine(regression_config, indicator_type)

result = engine.full_decision(
    X_eval=X_all,
    X_reg=X_train,
    y_reg=y_train
)
